# Практична робота №4
## Виявлення поведінкових аномалій та security-інцидентів в IoT-даних

У цій роботі потрібно перейти від звичайної batch/SQL-аналітики до інтерпретації поведінки IoT-пристроїв.

Ви працюєте з уже очищеним набором подій і результатами попередньої SQL-аналітики. Мета роботи — побудувати прості rule-based правила, знайти підозрілі часові вікна та сформувати таблицю findings.

### Вхідні дані

- `results/practical_02/clean_iot_events.parquet`
- `results/practical_03/device_activity_summary.csv`
- `results/practical_03/hourly_metric_summary.csv`
- `data/input/device_registry.csv`
- `data/input/metric_catalog.csv`
- `data/input/device_type_metrics.csv`
- `data/input/metadata.json`

### Результати роботи

Після виконання ноутбука мають бути створені файли:

- `results/practical_04/device_behavior_baseline.csv`
- `results/practical_04/hourly_anomaly_scores.csv`
- `results/practical_04/detection_findings.csv`
- `results/practical_04/practical_04_summary.json`

> У цій роботі не використовується приватний файл `injected_issues.csv`. Його немає у студентському наборі даних.


## Дані студента

Заповніть інформацію про себе.


In [ ]:
STUDENT_NAME = "TODO: Прізвище Ім'я По батькові"
GROUP = "TODO: група"
WORK_DATE = "TODO: дата виконання"


## 1. Підготовка середовища

Цей блок задає шляхи до даних і результатів. Його не потрібно змінювати, якщо структура проєкту не змінювалась.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import duckdb
import polars as pl


In [ ]:
ROOT = Path("/workspace") if Path("/workspace").exists() else Path.cwd().parent

INPUT_DIR = ROOT / "data" / "input"
PRACTICAL_02_RESULTS_DIR = ROOT / "results" / "practical_02"
PRACTICAL_03_RESULTS_DIR = ROOT / "results" / "practical_03"
PRACTICAL_04_RESULTS_DIR = ROOT / "results" / "practical_04"
PRACTICAL_04_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_EVENTS_PATH = PRACTICAL_02_RESULTS_DIR / "clean_iot_events.parquet"
DEVICE_ACTIVITY_PATH = PRACTICAL_03_RESULTS_DIR / "device_activity_summary.csv"
HOURLY_METRIC_PATH = PRACTICAL_03_RESULTS_DIR / "hourly_metric_summary.csv"

METADATA_PATH = INPUT_DIR / "metadata.json"
DEVICE_REGISTRY_PATH = INPUT_DIR / "device_registry.csv"
METRIC_CATALOG_PATH = INPUT_DIR / "metric_catalog.csv"
DEVICE_TYPE_METRICS_PATH = INPUT_DIR / "device_type_metrics.csv"

BASELINE_OUTPUT_PATH = PRACTICAL_04_RESULTS_DIR / "device_behavior_baseline.csv"
HOURLY_ANOMALY_OUTPUT_PATH = PRACTICAL_04_RESULTS_DIR / "hourly_anomaly_scores.csv"
FINDINGS_OUTPUT_PATH = PRACTICAL_04_RESULTS_DIR / "detection_findings.csv"
SUMMARY_OUTPUT_PATH = PRACTICAL_04_RESULTS_DIR / "practical_04_summary.json"

ROOT


In [ ]:
required_files = [
    CLEAN_EVENTS_PATH,
    DEVICE_ACTIVITY_PATH,
    HOURLY_METRIC_PATH,
    METADATA_PATH,
    DEVICE_REGISTRY_PATH,
    METRIC_CATALOG_PATH,
    DEVICE_TYPE_METRICS_PATH,
]

missing_files = [path for path in required_files if not path.is_file()]

if missing_files:
    message = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Практична робота №4 потребує завершених ПР2 і ПР3. "
        "Не знайдено такі файли:\n" + message
    )

print("Усі потрібні файли знайдено.")


## 2. Завантаження даних

У цьому блоці дані читаються у Polars DataFrame. Також створюється числова колонка `value_num`, щоб зручніше працювати з telemetry-метриками.


In [ ]:
with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

clean_events = pl.read_parquet(CLEAN_EVENTS_PATH)
device_registry = pl.read_csv(DEVICE_REGISTRY_PATH)
metric_catalog = pl.read_csv(METRIC_CATALOG_PATH)
device_type_metrics = pl.read_csv(DEVICE_TYPE_METRICS_PATH)

device_activity_summary = pl.read_csv(DEVICE_ACTIVITY_PATH)
hourly_metric_summary = pl.read_csv(HOURLY_METRIC_PATH)

if clean_events.schema.get("event_ts") == pl.Utf8:
    clean_events = clean_events.with_columns(
        pl.col("event_ts").str.to_datetime(strict=False).alias("event_ts")
    )

clean_events = clean_events.with_columns(
    pl.col("value").cast(pl.Float64, strict=False).alias("value_num")
)

device_registry = device_registry.with_columns(
    pl.col("expected_interval_sec").cast(pl.Float64, strict=False)
)

metric_catalog = metric_catalog.with_columns(
    [
        pl.col("value_type").str.to_lowercase().alias("value_type"),
        pl.col("min_value").cast(pl.Float64, strict=False),
        pl.col("max_value").cast(pl.Float64, strict=False),
    ]
)

print(f"Подій після очищення: {clean_events.height}")
print(f"Зареєстрованих пристроїв: {device_registry.height}")
print(f"Метрик у каталозі: {metric_catalog.height}")


In [ ]:
clean_events.head()


## 3. Схеми результатів і допоміжні функції

Усі findings мають бути зведені до єдиного формату. Не змінюйте назви колонок у `FINDINGS_COLUMNS`, бо фінальна перевірка очікує саме цю структуру.


In [ ]:
BASELINE_COLUMNS = [
    "device_id",
    "device_type",
    "location",
    "expected_interval_sec",
    "expected_events_per_hour",
    "observed_events_count",
    "first_event_ts",
    "last_event_ts",
    "observed_active_hours",
    "observed_events_per_hour",
]

FINDINGS_COLUMNS = [
    "finding_id",
    "scenario_type",
    "device_id",
    "device_type",
    "location",
    "start_ts",
    "end_ts",
    "metric",
    "score",
    "severity",
    "evidence",
]

PARTIAL_FINDINGS_COLUMNS = [column for column in FINDINGS_COLUMNS if column != "finding_id"]

PARTIAL_FINDINGS_SCHEMA = {
    "scenario_type": pl.Utf8,
    "device_id": pl.Utf8,
    "device_type": pl.Utf8,
    "location": pl.Utf8,
    "start_ts": pl.Utf8,
    "end_ts": pl.Utf8,
    "metric": pl.Utf8,
    "score": pl.Float64,
    "severity": pl.Utf8,
    "evidence": pl.Utf8,
}


In [ ]:
def empty_partial_findings() -> pl.DataFrame:
    return pl.DataFrame(schema=PARTIAL_FINDINGS_SCHEMA)


def normalize_partial_findings(df: pl.DataFrame) -> pl.DataFrame:
    if df.is_empty() and not df.columns:
        return empty_partial_findings()

    result = df

    for column, dtype in PARTIAL_FINDINGS_SCHEMA.items():
        if column not in result.columns:
            result = result.with_columns(pl.lit(None, dtype=dtype).alias(column))

    return result.select(
        [
            pl.col(column).cast(dtype, strict=False).alias(column)
            for column, dtype in PARTIAL_FINDINGS_SCHEMA.items()
        ]
    )


def add_finding_ids(df: pl.DataFrame) -> pl.DataFrame:
    normalized = normalize_partial_findings(df)

    if normalized.is_empty():
        return pl.DataFrame(schema={"finding_id": pl.Utf8, **PARTIAL_FINDINGS_SCHEMA})

    return (
        normalized.sort(["scenario_type", "device_id", "start_ts", "metric"])
        .with_row_index("finding_number", offset=1)
        .with_columns(
            pl.col("finding_number")
            .map_elements(lambda value: f"f-{value:05d}", return_dtype=pl.Utf8)
            .alias("finding_id")
        )
        .drop("finding_number")
        .select(FINDINGS_COLUMNS)
    )


def assert_has_columns(df: pl.DataFrame, expected_columns: list[str], df_name: str) -> None:
    missing = [column for column in expected_columns if column not in df.columns]
    if missing:
        raise AssertionError(f"{df_name}: відсутні колонки {missing}")


def counts_to_dict(df: pl.DataFrame, key_column: str) -> dict[str, int]:
    if df.is_empty():
        return {}

    return {
        str(key): int(count)
        for key, count in (
            df.group_by(key_column)
            .agg(pl.len().alias("count"))
            .sort(key_column)
            .iter_rows()
        )
    }


## 4. Побудова baseline поведінки пристроїв

Спочатку потрібно побудувати таблицю базової активності пристроїв. Вона знадобиться для подальших правил.

Очікуваний файл:

`results/practical_04/device_behavior_baseline.csv`

Мінімально таблиця має містити колонки з `BASELINE_COLUMNS`.


In [ ]:
def build_device_behavior_baseline(
    events: pl.DataFrame,
    devices: pl.DataFrame,
) -> pl.DataFrame:
    """Побудувати baseline активності кожного пристрою.

    TODO:
    1. Для кожного `device_id` порахуйте:
       - кількість подій;
       - перший timestamp;
       - останній timestamp;
       - тривалість активного періоду в годинах;
       - середню кількість подій на годину.
    2. Об'єднайте результат із `device_registry`.
    3. Додайте `expected_events_per_hour = 3600 / expected_interval_sec`.
    4. Поверніть DataFrame з колонками `BASELINE_COLUMNS`.
    """

    # TODO: реалізуйте побудову baseline.
    raise NotImplementedError("TODO: implement build_device_behavior_baseline")


device_behavior_baseline = build_device_behavior_baseline(clean_events, device_registry)

assert_has_columns(device_behavior_baseline, BASELINE_COLUMNS, "device_behavior_baseline")
device_behavior_baseline.write_csv(BASELINE_OUTPUT_PATH)

device_behavior_baseline.head()


## 5. Правило 1 — `missing_telemetry`

Ідея: якщо пристрій довго не надсилав жодних подій, це може означати втрату зв'язку, збій живлення або відмову пристрою.

Підказка: порівнюйте gap між сусідніми подіями одного пристрою з очікуваним інтервалом `expected_interval_sec`.


In [ ]:
MISSING_TELEMETRY_MULTIPLIER = 10
MISSING_TELEMETRY_MIN_GAP_SEC = 3600


def detect_missing_telemetry(
    events: pl.DataFrame,
    devices: pl.DataFrame,
) -> pl.DataFrame:
    """Знайти довгі прогалини у телеметрії пристроїв.

    TODO:
    1. Відсортуйте події за `device_id`, `event_ts`.
    2. Для кожного пристрою знайдіть попередній timestamp.
    3. Порахуйте `gap_sec`.
    4. Побудуйте threshold:
       `max(MISSING_TELEMETRY_MIN_GAP_SEC, expected_interval_sec * MISSING_TELEMETRY_MULTIPLIER)`.
    5. Сформуйте findings у форматі `PARTIAL_FINDINGS_COLUMNS`.
    """

    # TODO: реалізуйте правило missing_telemetry.
    return empty_partial_findings()


missing_telemetry_findings = detect_missing_telemetry(clean_events, device_registry)
normalize_partial_findings(missing_telemetry_findings).head()


## 6. Правило 2 — `sensor_stuck_value`

Ідея: якщо числова метрика довго має абсолютно однакове значення, сенсор міг “залипнути”.

Підказка: не всі числові метрики однаково корисні для цього правила. Бінарні або дискретні значення можуть створювати false positives.


In [ ]:
STUCK_MIN_EVENTS = 3
STUCK_MIN_DURATION_HOURS = 3


def detect_sensor_stuck_value(
    events: pl.DataFrame,
    devices: pl.DataFrame,
    metrics: pl.DataFrame,
) -> pl.DataFrame:
    """Знайти довгі послідовності однакових значень сенсора.

    TODO:
    1. Візьміть telemetry-події.
    2. Оберіть метрики, для яких правило має сенс.
    3. Для кожного `device_id + metric` знайдіть послідовності однакових `value_num`.
    4. Відійдіть від одиничних збігів: використайте мінімальну кількість подій і мінімальну тривалість.
    5. Сформуйте findings.
    """

    # TODO: реалізуйте правило sensor_stuck_value.
    return empty_partial_findings()


sensor_stuck_findings = detect_sensor_stuck_value(
    clean_events,
    device_registry,
    metric_catalog,
)
normalize_partial_findings(sensor_stuck_findings).head()


## 7. Правило 3 — `device_flooding`

Ідея: якщо пристрій у короткому часовому вікні надсилає значно більше подій, ніж очікується, це може бути flooding або помилка firmware.

Підказка: зручно аналізувати 15-хвилинні вікна.


In [ ]:
FLOOD_WINDOW_MINUTES = 15
FLOOD_MULTIPLIER = 5
FLOOD_MIN_EVENTS = 20


def detect_device_flooding(
    events: pl.DataFrame,
    devices: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Знайти вікна з надмірною кількістю подій.

    TODO:
    1. Створіть колонку `window_start`, округливши `event_ts` до 15-хвилинного вікна.
    2. Порахуйте кількість подій у кожному `device_id + window_start`.
    3. Порівняйте її з очікуваною кількістю подій у такому вікні.
    4. Сформуйте findings.
    5. Додатково поверніть проміжну таблицю scores для `hourly_anomaly_scores.csv`.
    """

    # TODO: реалізуйте правило device_flooding.
    empty_scores = pl.DataFrame(
        {
            "device_id": [],
            "device_type": [],
            "location": [],
            "window_start": [],
            "metric": [],
            "events_count": [],
            "baseline_value": [],
            "observed_value": [],
            "score": [],
            "scenario_hint": [],
        }
    )
    return empty_partial_findings(), empty_scores


device_flooding_findings, flood_scores = detect_device_flooding(clean_events, device_registry)
normalize_partial_findings(device_flooding_findings).head()


## 8. Правило 4 — `battery_drain`

Ідея: швидке падіння `battery_level` за кілька годин може означати проблему живлення або несправність пристрою.

Це правило складніше за попередні, тому можна реалізувати його простіше: для кожного пристрою порівняйте рівень батареї на початку і в кінці кількох часових вікон.


In [ ]:
BATTERY_MIN_DROP_POINTS = 20
BATTERY_MAX_WINDOW_HOURS = 12


def detect_battery_drain(
    events: pl.DataFrame,
    devices: pl.DataFrame,
) -> pl.DataFrame:
    """Знайти швидке падіння battery_level.

    TODO:
    1. Візьміть тільки `metric == "battery_level"`.
    2. Для кожного пристрою порівняйте значення батареї у часових вікнах до 12 годин.
    3. Якщо падіння перевищує `BATTERY_MIN_DROP_POINTS`, створіть finding.
    4. У `evidence` додайте початковий рівень, кінцевий рівень, drop і duration.
    """

    # TODO: реалізуйте правило battery_drain.
    return empty_partial_findings()


battery_drain_findings = detect_battery_drain(clean_events, device_registry)
normalize_partial_findings(battery_drain_findings).head()


## 9. Правило 5 — `data_poisoning`

Ідея: `data_poisoning` — це не обов'язково вихід за допустимі межі. Значення можуть залишатися валідними, але систематично зміщуватися протягом короткого часу.

Рекомендований підхід:

1. Працюйте тільки з telemetry-подіями та числовими метриками.
2. Порівнюйте поведінку `device_id + metric` у певній годині з baseline.
3. Baseline можна будувати:
   - або для `device_type + metric`;
   - або локально для цього ж `device_id + metric` у сусідніх годинах.
4. Сформуйте не всі проміжні відхилення, а тільки найбільш переконливі findings.


In [ ]:
POISONING_MIN_EVENTS_PER_HOUR = 3


def detect_data_poisoning(
    events: pl.DataFrame,
    devices: pl.DataFrame,
    metrics: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Знайти підозріле зміщення значень метрик.

    TODO:
    1. Візьміть telemetry-події з числовими значеннями.
    2. Оберіть метрики, для яких poisoning detection має сенс.
    3. Агрегуйте значення по `device_id + metric + hour`.
    4. Побудуйте baseline.
    5. Порахуйте score відхилення.
    6. Відфільтруйте найбільш підозрілі короткі вікна.
    7. Поверніть:
       - findings;
       - проміжну таблицю scores для `hourly_anomaly_scores.csv`.
    """

    # TODO: реалізуйте правило data_poisoning.
    empty_scores = pl.DataFrame(
        {
            "device_id": [],
            "device_type": [],
            "location": [],
            "hour": [],
            "metric": [],
            "events_count": [],
            "baseline_value": [],
            "observed_value": [],
            "score": [],
            "scenario_hint": [],
        }
    )
    return empty_partial_findings(), empty_scores


data_poisoning_findings, poisoning_scores = detect_data_poisoning(
    clean_events,
    device_registry,
    metric_catalog,
)
normalize_partial_findings(data_poisoning_findings).head()


## 10. Формування `hourly_anomaly_scores.csv`

Цей файл містить проміжні оцінки аномальності. Його можна використати для перевірки власної логіки та короткого аналізу в звіті.


In [ ]:
def build_hourly_anomaly_scores(
    flood_scores: pl.DataFrame,
    poisoning_scores: pl.DataFrame,
) -> pl.DataFrame:
    """Об'єднати проміжні anomaly scores.

    TODO:
    1. Приведіть проміжні scores з різних правил до спільного формату.
    2. Залиште найбільш корисні / найбільш підозрілі рядки.
    3. Поверніть DataFrame з колонками:
       device_id, device_type, location, hour, metric,
       events_count, baseline_value, observed_value, score, scenario_hint.
    """

    # TODO: реалізуйте формування hourly_anomaly_scores.
    return pl.DataFrame(
        {
            "device_id": [],
            "device_type": [],
            "location": [],
            "hour": [],
            "metric": [],
            "events_count": [],
            "baseline_value": [],
            "observed_value": [],
            "score": [],
            "scenario_hint": [],
        }
    )


hourly_anomaly_scores = build_hourly_anomaly_scores(flood_scores, poisoning_scores)
hourly_anomaly_scores.write_csv(HOURLY_ANOMALY_OUTPUT_PATH)

hourly_anomaly_scores.head()


## 11. Об'єднання findings

Коли всі правила реалізовані, об'єднайте їх в один файл `detection_findings.csv`.


In [ ]:
all_partial_findings = pl.concat(
    [
        normalize_partial_findings(missing_telemetry_findings),
        normalize_partial_findings(sensor_stuck_findings),
        normalize_partial_findings(device_flooding_findings),
        normalize_partial_findings(battery_drain_findings),
        normalize_partial_findings(data_poisoning_findings),
    ],
    how="diagonal_relaxed",
)

detection_findings = add_finding_ids(all_partial_findings)

assert_has_columns(detection_findings, FINDINGS_COLUMNS, "detection_findings")

if detection_findings.is_empty():
    raise AssertionError(
        "detection_findings порожній. Перевірте реалізацію detection-правил."
    )

detection_findings.write_csv(FINDINGS_OUTPUT_PATH)
detection_findings


## 12. Summary JSON

Фінальний JSON потрібен для швидкої перевірки результатів і короткого підсумку роботи.


In [ ]:
summary = {
    "variant_id": metadata.get("variant_id"),
    "student_id": metadata.get("student_id"),
    "student_name": metadata.get("student_name"),
    "baseline_rows": int(device_behavior_baseline.height),
    "hourly_anomaly_rows": int(hourly_anomaly_scores.height),
    "findings_total": int(detection_findings.height),
    "findings_by_scenario": counts_to_dict(detection_findings, "scenario_type"),
    "findings_by_severity": counts_to_dict(detection_findings, "severity"),
    "period_start": str(clean_events.select(pl.min("event_ts")).item()),
    "period_end": str(clean_events.select(pl.max("event_ts")).item()),
}

with SUMMARY_OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

summary


## 13. Фінальна перевірка результатів

Цей блок перевіряє, що всі очікувані файли створені та мають потрібну базову структуру.


In [ ]:
expected_outputs = [
    BASELINE_OUTPUT_PATH,
    HOURLY_ANOMALY_OUTPUT_PATH,
    FINDINGS_OUTPUT_PATH,
    SUMMARY_OUTPUT_PATH,
]

for path in expected_outputs:
    if not path.is_file():
        raise AssertionError(f"Не створено файл: {path}")
    if path.stat().st_size == 0:
        raise AssertionError(f"Файл порожній: {path}")

saved_findings = pl.read_csv(FINDINGS_OUTPUT_PATH)
assert_has_columns(saved_findings, FINDINGS_COLUMNS, "saved_findings")

saved_baseline = pl.read_csv(BASELINE_OUTPUT_PATH)
assert_has_columns(saved_baseline, BASELINE_COLUMNS, "saved_baseline")

print("Фінальна перевірка пройдена.")


## 14. Висновок

Напишіть короткий висновок українською мовою.

У висновку потрібно описати:

1. Які типи підозрілої поведінки було знайдено.
2. Які правила дали найбільше findings.
3. Які findings виглядають найбільш переконливо.
4. Де можливі false positives.
5. Чому rule-based detection не гарантує ідеального відновлення прихованих сценаріїв.


TODO: напишіть висновок тут.
